# Find where each channel leaks

*Data Analysis in Python · Topic 11.1*

Your media team wants to know why Paid Social and Paid Video convert so badly: do their visitors bounce, or do they stop later? Build the session funnel from the raw events, by channel group, and find the channel whose item views turn into carts least often.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that reads the session starts and the funnel steps from `events.parquet`, and the cell that builds the `channel_groups` table: the channel group of each source and medium, with Foundations topic 12.3.'s function.
2. In the first cell marked ✏️, create the `sessions` table: the `events` table's `session_start` rows without the `event_name` column, one per session, with the `channel_group` column from the `channel_groups` table.
3. In the second cell marked ✏️, add a column to the `sessions` table for each step in `STEPS`: `True` if the session has at least one event of that name, and `False` if it has none. Identify a session by `user_pseudo_id` and `ga_session_id` together.
4. In the last cell marked ✏️, create the `funnel` table: one row per channel group, with the columns `sessions`, `view_item`, `add_to_cart`, `begin_checkout`, `purchase` and `conversion_rate`. Each step's column is its rate from the step before it, `view_item`'s from the sessions, and `conversion_rate` is the purchases' share of the sessions. Then create `weakest_cart_channel`: the channel group with the lowest `add_to_cart` rate.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `sessions` table has 2,506,005 rows, and 1,331,075 of them viewed an item. The `funnel` table has 9 channel groups: Email converts 6.30% of its sessions and Paid Video 0.65%. `weakest_cart_channel` is `Paid Video`, whose add-to-cart rate is 3.93%.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
STEPS = ["view_item", "add_to_cart", "begin_checkout", "purchase"]

In [ ]:
# The session starts and the funnel steps from the event stream, with the categories as text. Run it, and leave it as it is.
events = pd.read_parquet(DATA_URL + "events.parquet",
                         columns=["event_name", "user_pseudo_id", "ga_session_id", "device_category", "source", "medium"],
                         filters=[("event_name", "in", ["session_start"] + STEPS)])
events = events.astype({"event_name": str, "device_category": str, "source": str, "medium": str})
events.shape

In [ ]:
# The channel group of each source and medium, with Foundations topic 12.3's function. Run it, and leave it as it is.
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium, with Foundations topic 12.3's rules."""
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"


# Each pair once, so the function runs 23 times instead of 2.5 million.
channel_groups = events[["source", "medium"]].drop_duplicates().reset_index(drop=True)
channel_groups["channel_group"] = [channel_group(source, medium) for source, medium in zip(channel_groups["source"], channel_groups["medium"])]
channel_groups.shape

## One row per session

In [ ]:
sessions = events[events["event_name"] == "session_start"].drop(columns="event_name")
sessions = sessions.merge(channel_groups, on=["source", "medium"], how="left", validate="many_to_one")
sessions.shape

## The steps each session reached

In [ ]:
step_events = events[events["event_name"] != "session_start"]
step_counts = pd.crosstab([step_events["user_pseudo_id"], step_events["ga_session_id"]], step_events["event_name"])
sessions = sessions.merge(step_counts[STEPS], on=["user_pseudo_id", "ga_session_id"], how="left", validate="one_to_one")
sessions[STEPS] = sessions[STEPS].fillna(0) > 0
sessions[STEPS].sum()

## The funnel by channel group

In [ ]:
counts = sessions.groupby("channel_group")[STEPS].sum()
counts["sessions"] = sessions.groupby("channel_group").size()
funnel = pd.DataFrame({
    "sessions": counts["sessions"],
    "view_item": counts["view_item"] / counts["sessions"],
    "add_to_cart": counts["add_to_cart"] / counts["view_item"],
    "begin_checkout": counts["begin_checkout"] / counts["add_to_cart"],
    "purchase": counts["purchase"] / counts["begin_checkout"],
    "conversion_rate": counts["purchase"] / counts["sessions"],
})
weakest_cart_channel = funnel["add_to_cart"].idxmin()
funnel.round(4), weakest_cart_channel

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    steps = ["view_item", "add_to_cart", "begin_checkout", "purchase"]
    expected = {
        # channel group: sessions, view_item, add_to_cart, begin_checkout, purchase, conversion_rate
        "Direct": (369978, 0.537905, 0.158975, 0.648429, 0.736729, 0.040851),
        "Display": (143999, 0.526622, 0.044941, 0.529049, 0.613422, 0.007681),
        "Email": (202255, 0.548189, 0.207903, 0.705219, 0.783219, 0.062950),
        "Organic Search": (602514, 0.530477, 0.127805, 0.589586, 0.687469, 0.027480),
        "Organic Social": (125800, 0.529467, 0.094585, 0.620794, 0.721810, 0.022440),
        "Paid Search": (386561, 0.526755, 0.155331, 0.520978, 0.604139, 0.025753),
        "Paid Social": (361273, 0.526145, 0.065956, 0.516870, 0.603395, 0.010823),
        "Paid Video": (241035, 0.527591, 0.039310, 0.520104, 0.603462, 0.006509),
        "Referral": (72590, 0.527001, 0.110469, 0.517037, 0.613272, 0.018460),
    }

    sessions = found.get("sessions")
    if not isinstance(sessions, pd.DataFrame):
        results.append((False, "sessions should be a DataFrame: the events table's session_start rows, one per session."))
    elif len(sessions) == 1331075:
        results.append((False, "The sessions table has 1,331,075 rows: only the sessions that viewed an item. Join the step counts with how=\"left\", so the sessions that reached no step stay in."))
    elif len(sessions) != 2506005:
        results.append((False, f"The sessions table should have 2,506,005 rows, one per session_start event. It has {len(sessions):,}: check that each join matches a session once, with validate=."))
    elif "channel_group" not in sessions.columns:
        results.append((False, "The sessions table should have a channel_group column, from the channel_groups table, joined on source and medium."))
    elif (sessions["channel_group"] == "Email").sum() != 202255 or sessions["channel_group"].isna().any():
        results.append((False, "The sessions table's channel_group should come from the channel_groups table, joined on both source and medium: 202,255 sessions are Email."))
    else:
        results.append((True, "2,506,005 sessions, one per user and session ID, each with its channel group."))

    if isinstance(sessions, pd.DataFrame) and len(sessions) == 2506005:
        missing = [step for step in steps if step not in sessions.columns]
        if missing:
            results.append((False, f"The sessions table should have a column for each step. It doesn't have {', '.join(missing)}."))
        elif not all(pd.api.types.is_bool_dtype(sessions[step]) for step in steps):
            results.append((False, "The step columns should be True or False: did the session reach the step? Turn the counts into that with .fillna(0) > 0."))
        else:
            reached = [int(sessions[step].sum()) for step in steps]
            if reached[0] == 1366847:
                results.append((False, "1,366,847 sessions viewed an item: too many, because the step counts are joined on ga_session_id alone. Two users who start a session in the same second share one. Count and join on user_pseudo_id and ga_session_id together."))
            elif reached != [1331075, 158637, 94312, 65106]:
                results.append((False, f"The step columns should count 1,331,075, 158,637, 94,312 and 65,106 sessions. They count {', '.join(f'{n:,}' for n in reached)}."))
            else:
                results.append((True, "1,331,075 sessions viewed an item, 158,637 added to the cart, 94,312 began the checkout and 65,106 bought."))

    funnel = found.get("funnel")
    columns = ["sessions"] + steps + ["conversion_rate"]
    if not isinstance(funnel, pd.DataFrame):
        results.append((False, "funnel should be a DataFrame: one row per channel group."))
    elif set(funnel.index) != set(expected):
        results.append((False, f"funnel's index should be the 9 channel groups. It is {list(funnel.index)[:10]}."))
    elif [column for column in columns if column not in funnel.columns]:
        results.append((False, f"funnel should have the columns {', '.join(columns)}. It doesn't have {', '.join(column for column in columns if column not in funnel.columns)}."))
    elif abs(funnel.loc["Email", "add_to_cart"] - 0.113970) < 0.00001:
        results.append((False, "funnel's add_to_cart for Email is 0.1140: the share of all sessions. Divide each step by the step before it: add_to_cart by view_item."))
    else:
        wrong = [(group, column) for group, values in expected.items() for column, value in zip(columns, values)
                 if not pd.api.types.is_number(funnel.loc[group, column]) or abs(funnel.loc[group, column] - value) > (0.5 if column == "sessions" else 0.00001)]
        if wrong:
            group, column = wrong[0]
            value = expected[group][columns.index(column)]
            results.append((False, f"funnel's {column} for {group} should be {value:,.6g}. It is {funnel.loc[group, column]!r}. Each step's rate is its sessions divided by the step before it's."))
        else:
            results.append((True, "Email converts 6.30% of its sessions and Paid Video 0.65%. They view items about as often, 54.8% and 52.8%: the gap opens at the cart."))

    weakest = found.get("weakest_cart_channel")
    if weakest == "Paid Video":
        results.append((True, "Paid Video: only 3.9% of its item views go into the cart, against Email's 20.8%. Its visitors look at the products and leave."))
    elif weakest == "Email":
        results.append((False, "weakest_cart_channel is Email, the strongest. Use idxmin(), not idxmax()."))
    else:
        results.append((False, f"weakest_cart_channel should be the channel group with the lowest add_to_cart rate: funnel[\"add_to_cart\"].idxmin(). It is {weakest!r}."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
sessions = events[events["event_name"] == "session_start"].drop(columns="event_name")
sessions = sessions.merge(channel_groups, on=["source", "medium"], how="left", validate="many_to_one")
sessions.shape
```

```python
step_events = events[events["event_name"] != "session_start"]
step_counts = pd.crosstab([step_events["user_pseudo_id"], step_events["ga_session_id"]], step_events["event_name"])
sessions = sessions.merge(step_counts[STEPS], on=["user_pseudo_id", "ga_session_id"], how="left", validate="one_to_one")
sessions[STEPS] = sessions[STEPS].fillna(0) > 0
sessions[STEPS].sum()
```

```python
counts = sessions.groupby("channel_group")[STEPS].sum()
counts["sessions"] = sessions.groupby("channel_group").size()
funnel = pd.DataFrame({
    "sessions": counts["sessions"],
    "view_item": counts["view_item"] / counts["sessions"],
    "add_to_cart": counts["add_to_cart"] / counts["view_item"],
    "begin_checkout": counts["begin_checkout"] / counts["add_to_cart"],
    "purchase": counts["purchase"] / counts["begin_checkout"],
    "conversion_rate": counts["purchase"] / counts["sessions"],
})
weakest_cart_channel = funnel["add_to_cart"].idxmin()
funnel.round(4), weakest_cart_channel
```

</details>